# 주식 트레이딩 - DQN사용

In [10]:
import math, random
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# 데이터 준비 (Close 칼럼 - 종가데이터)
def load_return(csv_path:str):
  df = pd.read_csv(csv_path)
  close = df['Close'].astype(float).values
  ret = np.zeros_like(close, dtype=np.float32) # 같은 길이의 수익률 배열 초기화
  # 일별 수익률
  ret[1:] = (close[1:] - close[:-1]) / (close[:-1] + 1e-9) # 분모가 0이 되는것을 방지
  return ret

# 환경
class TradingEnv:
  # 매개변수 : self, 수익률 배열, 관측일, 거래 비용
  def __init__(self, returns:np.ndarray, window=20, cost_bps=10.0):
    assert len(returns) > window + 1, '데이터가 너무 짧아요'

    self.rets_all = returns.astype(np.float32)  # 수익률 배열값 넘겨받는 변수 생성
    self.window = window                        # 관측에 사용할 과거 수익률 갯수
    self.cost = cost_bps / 10_000.0             # 거래 비용 실수 비율로 변환 (bps = 1 / 100bp)
    self.reset()

  # 장식자를 줘서 메소드를 변수처럼 접근 가능하게함.
  # 호출 : 객체 변수명.obs_dim <- ()를 안써도 됨.
  @property
  def obs_dim(self):
    return self.window + 1 # 관측 차원 : 최근 window일 수익률 + 현재 position 의 스칼라값을 하나 줌.

  @property
  def n_avctions(self):
    return 3 # 행동 공간 크기 (숏(매도):-1, 현금:0, 롱(매수):1)

  def reset(self): # overriding - 에피소드 호출시 자동 실행되는 메소드
    self.t = self.window  # t를 window부터 시작
    self.pos = 0          # 초기 position은 0에서 출발
    return self._obs()     # 현재 관측치를 반환

  def _obs(self): # 현재 시점의 관측 벡터 생성
    window = self.rets_all[self.t - self.window:self.t] # 직전 window일의 수익률 슬라이싱
    return np.concatenate([window, [float(self.pos)]]).astype(np.float32) # 수익률들 + 현재 포지션 반환

  def step(self, action:int): # 환경 한 스텝 진행(행동에 의해 다음상태/보상/종료)
    new_pos = [-1, 0, 1][action] # 행동 인덱스를 실제 포지션 값으로 매핑 ( 0 -> -1, 1 -> 0, 2 -> 1)
    trade_cost = self.cost * abs(new_pos - self.pos) # 거래 비용 position 변화량
    reward = self.pos * self.rets_all[self.t] - trade_cost # 보상 ((이전position * 금일 수익률) - 거래비용)
    self.pos = new_pos # state(position) 갱신
    self.t += 1 # 날짜 갱신
    done = (self.t >= len(self.rets_all)) # 마지막 전날 까지 도달하면 끝남.
    return self._obs(), float(reward), done

# Q-Network 구성하기
def build_qnet_seq(obs_dim, n_avctions):
  model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(obs_dim,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(n_avctions, activation='linear')
  ])
  return model

# Replay Buffer
class Replay:
  def __init__(self, cap=20000):
    self.buf = deque(maxlen=cap)

  def __len__(self):
    return len(self.buf)

  def push(self, *tr):
    self.buf.append(tr) # (obs, action, reward, nobs, done)

  def sample(self, n): # minibatch sampleing
    s = random.sample(self.buf, n) #
    s, a, r, n, d = zip(*s)
    return (np.array(s, np.float32),  # 상태 배열(obs_dim)
            np.array(a, np.float32),  # 행동 배열
            np.array(r, np.float32),  # 보상 배열
            np.array(n, np.float32),  # 다음 상태 배열
            np.array(d, np.float32)   # 종료 배열
    )

class DQN:
  # 생성자로 하이퍼파라미터 초기화
  def __init__(self, obs_dim, n_avctions, lr=3e-4, gamma=0.99, batch=128):
    self.q = build_qnet_seq(obs_dim, n_avctions) # Q-network
    self.tgt = build_qnet_seq(obs_dim, n_avctions) # target-network
    self.tgt.set_weights(self.q.get_weights()) # target-network의 가중치를 main과 맞추는 작업
    self.opt = tf.keras.optimizers.Adam(lr)
    self.gamma, self.batch = gamma, batch # 멤버필드로 전환
    self.buf = Replay() # Replay Buffer 객체 생성
    self.loss_fn = tf.keras.losses.Huber() # mse 보다 이상치에 덜 민감한 Huber 손실함수 사용
    self.eps = 1.0
    self.eps_decay = 0.9995
    self.eps_min = 0.05
    self.n_avctions = n_avctions

  # 행동을 선택하는 함수
  def act(self, obs):
    # 탐험
    if random.random() < self.eps:
      return random.randrange(self.n_avctions) # 무작위 행동 인덱스 반환

    # 이용
    qv = self.q(obs[None, :], training=False).numpy()[0] # Q(s, )계산
    return int(np.argmax(qv))

  # 파라미터 업데이트(갱신) - Replay Buffer에서 minibatch sampleing
  def update(self):
    # buffer에 sample이 충분 X
    if len(self.buf) < self.batch:
      return
    # buffer에 sample이 충분 O
    s, a, r, ns, d = self.buf.sample(self.batch) # minibatch sampleing
    a_oh = tf.one_hot(a, self.n_avctions) # 행동인덱스를 one-hot vector encoding
    '''
    참고 : DQN에서 GradientTape()로 학습할 때 행동 인덱스를 원핫 처리하는 이유는?
    모델이 출력한 여러 행동의 Q값 중에서 실제로 선택한 행동의 Q값만 골라서 손실 계산에 사용하기 위해서이다.
    '''
    with tf.GradientTape() as tape:
      q_sa = tf.reduce_sum(self.q(s) * a_oh, axis=1) # 내적 Q(s, a)
      q_next = tf.reduce_max(self.tgt(ns), axis=1) # Target Network로 다음 상태의 최대 Q값을 얻음
      y = r + (1 - d) * self.gamma * q_next # 타깃값 y = 벨만 방정식
      loss = self.loss_fn(y, q_sa) # 손실 = Huber(y, Q(s, a))

    # 경사하강법 계산
    g = tape.gradient(loss, self.q.trainable_variables)
    # 경사하강법 스텝 적용
    self.opt.apply_gradients(zip(g, self.q.trainable_variables))
    self.tgt.set_weights(self.q.get_weights())
    self.eps = max(self.eps * self.eps_decay, self.eps_min) # epsilon 갱신


# 학습 하기
def train_go(csv_path='prices.csv', window=20, cost_bps=10.0, episodes=5):
  rets = load_return(csv_path)
  # print(rets)
  # 환경 생성
  env = TradingEnv(rets, window=window, cost_bps=cost_bps)
  # DQN 생성
  agent = DQN(env.obs_dim, env.n_avctions) # 데코레이션 메소드 호출
  equity = [] # 누적 PnL 추적 리스트 (성과 지표용)

  for ep in range(episodes):
    obs = env.reset() # 환경 리셋
    done, ep_pnl = False, 0.0 # ep_pnl = 에피소드 누적 PnL 초기화

    # 학습하기
    while not done:
      act = agent.act(obs)
      # The issue is here: `env.step(act)` is called twice, causing `self.t` to increment twice for each agent action. This can lead to out-of-bounds errors or incorrect environment behavior.
      # Additionally, the first `env.step(act)` call's return values are not captured, and the `obs` variable used for `agent.buf.push` is the one *before* the step.
      # The corrected logic should be to call `env.step(act)` once and then use its returned `nobs, r, done` for the buffer and `ep_pnl` update.
      nobs, r , done = env.step(act) # 다음관측, 보상, 종료
      agent.buf.push(obs, act, r, nobs, float(done)) # Replay Buffer에 저장
      agent.update() # Main Network에서 학습을 함 - 파라미터 업데이트
      obs = nobs # Update current observation to the next observation
      ep_pnl += r # 에피소드내에서의 누적 PnL 갱신(보상의 합)
      equity.append(ep_pnl)

    print(f'ep:{ep} / {episodes} , PnL={ep_pnl:.4f}, eps={agent.eps:.3f}')

  # 결과 요약
  equity = np.array(equity)
  daily_ret = np.diff(equity, prepend=0.0) # 일별 PnL 증분(보상 시퀀스) 추정
  '''
  샤프비율(Sharpe Ratio)란? 수익률의 ‘위험 대비 성과’를 나타내는 지표
  일별 수익률 평균 / 일별 수익률 표준편차 * sqrt(252) - 1년 거래 일수 252로 봄
  '''
  sharp = daily_ret.mean() / (daily_ret.std() + 1e-9) * np.sqrt(252)
  print('--- 요약 결과 ---')
  print(f'Final PnL : {equity[-1]:.5f}')
  print(f'sharpe ratio : {sharp:.3f}')


if __name__ == '__main__':
  train_go(csv_path='prices.csv', window=20, cost_bps=10.0, episodes=5)


ep:0 / 5 , PnL=0.0044, eps=1.000
ep:1 / 5 , PnL=-0.0044, eps=1.000
ep:2 / 5 , PnL=0.0029, eps=1.000
ep:3 / 5 , PnL=-0.0119, eps=1.000
ep:4 / 5 , PnL=-0.0186, eps=1.000
--- 요약 결과 ---
Final PnL : -0.01855
sharpe ratio : -0.523
